# V103 - Bagged LightGBM

Pre-registered plan: bagging lifted Ridge by smoothing template dominance, and trees have higher variance than Ridge, so bagging should help trees more. Ten-bag LightGBM with out-of-bag predictions faces single LightGBM and bagged Ridge under grouped folds with a fresh seed. Same locks. If variance was all that held trees back, this shows it.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2828
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
rng = np.random.RandomState(SEED)
PARAMS = dict(objective="regression", n_estimators=300, learning_rate=0.05, num_leaves=31,
                min_data_in_leaf=50, verbose=-1, deterministic=True, seed=SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v103")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Bagged trees vs single trees vs bagged Ridge

Same word-only features everywhere. OOB predictions for both bagged contenders, grouped folds for the locks.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_oof(fit_predict_one, seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        acc[oob] += fit_predict_one(idx, oob)
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

def ridge_one(idx, oob):
    P = np.zeros((len(oob), len(MODS)))
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[idx], Y[idx, j])
        P[:, j] = m.predict(W[oob])
    return P

def lgbm_one(idx, oob):
    P = np.zeros((len(oob), len(MODS)))
    for j in range(len(MODS)):
        g = lgb.LGBMRegressor(**PARAMS)
        g.fit(W[idx], Y[idx, j])
        P[:, j] = g.predict(W[oob])
    return P

oof_br = bagged_oof(ridge_one, SEED)
oof_bg = bagged_oof(lgbm_one, SEED + 1)
oof_sg = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        g = lgb.LGBMRegressor(**PARAMS)
        g.fit(W[tr], Y[tr, j])
        oof_sg[va, j] = g.predict(W[va])
    fids[va] = fi + 1
sr, sg, sb = (ndcg5(Y, o) for o in (oof_br, oof_sg, oof_bg))
print(f"bag-ridge={sr.mean():.5f} single-gbm={sg.mean():.5f} bag-gbm={sb.mean():.5f}")
d = sb - sr
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_bg[fids == k]).mean() - sr[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"bag-gbm vs bag-ridge: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT bagged trees" if (lo > GATE_FLOOR and wf >= 0) else "KEEP bagged Ridge; trees stay behind even bagged")

bag-ridge=0.63856 single-gbm=0.60199 bag-gbm=0.61731
bag-gbm vs bag-ridge: delta=-0.02125 CI95=[-0.02547,-0.01710] worst-fold=-0.02693
KEEP bagged Ridge; trees stay behind even bagged
